# Cell type-wise KEGG/GO GSEA heatmap using ggplot2

Final version with empty columns removed, guarded SVG saving, region labels rotated 90 degrees, category legend shifted right with larger color boxes, and extra title spacing.

Key features:

- GO/KEGG terms are combined in one heatmap.
- Candidate-module columns with no curated term enrichment are removed.
- Heatmap tiles are drawn with `geom_tile()` for crisp PDF output.
- X-axis labels use one-line `celltype_region_Mx` names and are drawn directly under the heatmap.
- Region annotations are filled boxes matching the cell type strip; labels are rotated 90 degrees.
- Category legend is shown under the terms, shifted right to avoid mixing with term labels.
- SVG output is skipped gracefully if `svglite` is not installed.


In [ ]:
# ============================================================
# 1. Load packages
# ============================================================

# Run this once if packages are missing:
# install.packages(c("tidyverse", "ggnewscale", "cowplot", "scales", "grid"))

library(tidyverse)
library(ggnewscale)
library(cowplot)
library(scales)
library(grid)

In [ ]:
# ============================================================
# 2. User settings
# ============================================================

GO_FILE <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/05_gsea/results/go_bp/GO_positive_p005_hits.tsv"
KEGG_FILE <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/05_gsea/results/kegg/KEGG_positive_p005_hits.tsv"

OUTDIR <- "/project/zduren/durenlab/naqing_palmetto/REGA_separate/05_gsea/results/figures/celltype_wise_GSEA_heatmap"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)

OUT_PREFIX <- "Figure_celltype_wise_GO_KEGG_combined_heatmap"
TITLE <- "KEGG/GO enrichment across candidate regulatory modules"

REGION_ORDER <- c("AG", "EC", "HC", "MTC", "PFC", "TH")
CELLTYPE_ORDER <- c("Exc", "Inh", "Ast", "Oli", "OPC", "Mic_Immune")

Region = c(
    "AG"  = "#8DA0CB",
    "EC"  = "#FC8D62",
    "HC"  = "#66C2A5",
    "MTC" = "#E78AC3",
    "PFC" = "#B3B3B3",
    "TH"  = "#A6D854"
 )


celltype_colors <- c(
    "Exc"        = "#0072B2",
    "Inh"        = "#E69F00",
    "Ast"        = "#009E73",
    "Oli"        = "#D55E00",
    "OPC"        = "#CC79A7",
    "Mic_Immune" = "#6B5B53"#,
    #"Vasc"       = "#56B4E9"
)

category_colors <- c(
  "Neurodegenerative disease" = "#6A51A3",
  "Immune / inflammation" = "#CB181D",
  "Mitochondrial / proteostasis / metabolism / stress" = "#2171B5",
  "Neuronal / synaptic / structural processes" = "#228833"
)

In [ ]:
# ============================================================
# 3. Figure parameters
# ============================================================

# Controls heatmap cell size. coord_fixed(ratio = 1) keeps cells close to square.
CELL_SIZE <- 0.55

# Output size components, in inches.
LEFT_WIDTH <- 8.5
RIGHT_TEXT_UNITS <- 43     # x-axis coordinate units reserved for term labels + category legend
TOP_HEIGHT <- 3.8
BOTTOM_HEIGHT <- 8.0
MIN_FIG_WIDTH <- 50
MIN_FIG_HEIGHT <- 8

MAX_COLOR_VALUE <- 5.0

TITLE_FONTSIZE <- 60
REGION_FONTSIZE <- 55
REGION_TEXT_ANGLE <- 90
TERM_TEXT_SIZE <- 55
XTICK_FONTSIZE <- 55
LEGEND_FONTSIZE <- 55
COLORBAR_FONTSIZE <- 55
CATEGORY_LEGEND_TEXT_SIZE <- 60

# Category color bar on the left side of the heatmap.
# Right edge is exactly 0.5, the left boundary of the first heatmap tile.
CATEGORY_BAR_XMAX <- 0.5
CATEGORY_BAR_WIDTH <- 0.85

# Top annotation positions in heatmap coordinate units.
CELLTYPE_BAR_Y_OFFSET <- 0.55
CELLTYPE_BAR_HEIGHT <- 0.55
REGION_BAR_Y_OFFSET <- 1.15
REGION_BAR_HEIGHT <- CELLTYPE_BAR_HEIGHT
REGION_TEXT_Y_OFFSET <- 2.05

# Right-side term labels are drawn inside the same ggplot panel.
# Heatmap tile centers are 1:n_cols, so right edge is n_cols + 0.5.
TERM_LABEL_GAP <- 0.00
RIGHT_TERM_X <- NULL
RIGHT_LABEL_XMAX <- NULL

# Manually drawn x-axis labels. This keeps them next to the heatmap bottom
# even though the category legend extends below the heatmap.
X_LABEL_Y <- 0.35

# Category legend below the last term label.
# Shift right so color boxes do not mix with the last term labels.
CATEGORY_LEGEND_TOP_GAP <- 0.90
CATEGORY_LEGEND_ROW_GAP <- 1.15
CATEGORY_LEGEND_X_SHIFT <- 1.10
CATEGORY_LEGEND_BOX_SIZE <- 0.42
CATEGORY_LEGEND_TEXT_X <- 0.65

# Extra spacing between title and plot, to avoid overlap with region names.
TITLE_BOTTOM_MARGIN <- 35

# Main heatmap margins. Increase LEFT if left-side legends are clipped.
MAIN_MARGIN_TOP <- 10
MAIN_MARGIN_RIGHT <- 10
MAIN_MARGIN_BOTTOM <- 10
MAIN_MARGIN_LEFT <- 20

# Output switches.
WRITE_TABLES <- TRUE
CLUSTER_WITHIN_CATEGORY <- TRUE

# Make text editable in vector output.
options(bitmapType = "cairo")



In [ ]:
# ============================================================
# 4. Candidate modules
# ============================================================

CANDIDATES <- list(
  "Exc_AG" = c("M8"),
  "Inh_AG" = c("M8"),
  "Ast_AG" = c("M2"),
  "Oli_AG" = c("M6"),

  "Exc_EC" = c("M13"),
  "Ast_EC" = c("M17"),
  "Oli_EC" = c("M14", "M16"),
  "Mic_Immune_EC" = c("M16"),

  "Ast_HC" = c("M13"),
  "OPC_HC" = c("M13"),

  "Exc_MTC" = c("M8", "M19"),
  "Ast_MTC" = c("M8"),
  "Oli_MTC" = c("M8", "M13"),
  "OPC_MTC" = c("M8"),
  "Mic_Immune_MTC" = c("M8"),

  "Exc_PFC" = c("M1"),
  "Inh_PFC" = c("M1"),
  "Ast_PFC" = c("M4", "M17"),
  "OPC_PFC" = c("M14")
)

In [ ]:
TERM_CATEGORIES <- list(
  "Neurodegenerative disease" = c(
    "ALZHEIMER DISEASE",
    "PATHWAYS OF NEURODEGENERATION - MULTIPLE DISEASES",
    "PARKINSON DISEASE",
    "HUNTINGTON DISEASE",
    "PRION DISEASE",
    "AMYOTROPHIC LATERAL SCLEROSIS"
  ),

  "Immune / inflammation" = c(
    "ANTIGEN PROCESSING AND PRESENTATION",
    "CYTOKINE-CYTOKINE RECEPTOR INTERACTION",
    "COMPLEMENT AND COAGULATION CASCADES",
    "INFLAMMATORY BOWEL DISEASE",
    "AUTOIMMUNE THYROID DISEASE",
    "ALLOGRAFT REJECTION",
    "Positive Regulation of Immune System Process",
    "Regulation of T Cell Proliferation",
    "Antigen Processing and Presentation",
    #"Antigen Processing and Presentation of Exogenous Peptide Antigen",
    #"Antigen Processing and Presentation of Endogenous Peptide Antigen",
    "Peptide Antigen Assembly With MHC Protein Complex"
),

  "Mitochondrial / proteostasis / metabolism / stress" = c(
    "OXIDATIVE PHOSPHORYLATION",
    "RIBOSOME",
    "PROTEASOME",
    "Oxidative Phosphorylation",
    "Mitochondrial ATP Synthesis Coupled Electron Transport",
    "Mitochondrial Electron Transport, Cytochrome C to Oxygen",
    "Cellular Respiration",
    "ATP Synthesis Coupled Electron Transport",
    "Lipid Biosynthetic Process",
    "Protein Folding",
    "Proteasomal Protein Catabolic Process",
    "Cytoplasmic Translation",
    "Translation",
    "Ribosome Biogenesis"
  ),

  "Neuronal / synaptic / structural processes" = c(
    "NEUROACTIVE LIGAND-RECEPTOR INTERACTION",
    "CALCIUM SIGNALING PATHWAY",
    "GLUTAMATERGIC SYNAPSE",
    "GABAERGIC SYNAPSE",
    "CHOLINERGIC SYNAPSE",
    "DOPAMINERGIC SYNAPSE",
    "SEROTONERGIC SYNAPSE",
    "AXON GUIDANCE",
    "ECM-RECEPTOR INTERACTION",
    "CELL ADHESION MOLECULES",

    "Synapse Organization",
    "Synapse Assembly",
    "Trans-synaptic Signaling",
    "Chemical Synaptic Transmission",
    #"Regulation of Synaptic Transmission, Glutamatergic",
    #"Synaptic Transmission, GABAergic",
    "Axon Guidance",
    "Axonal Transport",
    "Calcium Ion Transport",
    "Extracellular Matrix Organization",
    "Cilium Organization"
  )
)


In [ ]:
# ============================================================
# 6. Helper functions
# ============================================================

module_number <- function(module) {
  m <- stringr::str_match(as.character(module), "(\\d+)")[, 2]
  ifelse(is.na(m), 999, as.integer(m))
}

clean_go_term <- function(term) {
  term %>%
    as.character() %>%
    stringr::str_remove("\\s*\\(GO:\\d+\\)\\s*$") %>%
    stringr::str_squish()
}

normalize_term <- function(term) {
  stringr::str_to_upper(clean_go_term(term))
}

pretty_term <- function(term) {
  term <- clean_go_term(term)
  term <- ifelse(term == stringr::str_to_upper(term), stringr::str_to_title(term), term)
  term <- stringr::str_replace_all(term, c(
    "Ad" = "AD",
    "Als" = "ALS",
    "Nf-Kappa" = "NF-kappa",
    "Jak-Stat" = "JAK-STAT",
    "Tca" = "TCA",
    "Ecm" = "ECM"
  ))
  term
}

parse_celltype_region <- function(celltype_region) {
  region <- stringr::str_extract(celltype_region, "[^_]+$")
  celltype <- stringr::str_remove(celltype_region, paste0("_", region, "$"))
  celltype <- ifelse(tolower(celltype) == "mic_immune", "Mic_Immune", celltype)
  tibble(celltype = celltype, region = region)
}

category_label_short <- function(category) {
  dplyr::recode(
    category,
    "Neurodegenerative disease" = "Neurodegenerative disease",
    "Immune / inflammation" = "Immune / inflammation",
    "Mitochondrial / proteostasis / metabolism / stress" = "Mitochondrial / proteostasis / metabolism / stress",
    "Neuronal / synaptic / structural processes" = "Neuronal / synaptic / structural processes",
    .default = category
  )
}

In [ ]:
# ============================================================
# 7. Build candidate column order
# ============================================================

build_column_order <- function() {
  purrr::imap_dfr(CANDIDATES, function(mods, celltype_region) {
    parsed <- parse_celltype_region(celltype_region)
    tibble(
      celltype_region = celltype_region,
      celltype = parsed$celltype,
      region = parsed$region,
      module = mods,
      key = paste(celltype_region, mods, sep = "_"),
      display = paste(celltype, parsed$region, mods, sep = "_")
    )
  }) %>%
    mutate(
      region = factor(region, levels = REGION_ORDER),
      celltype = factor(celltype, levels = CELLTYPE_ORDER),
      module_rank = module_number(module)
    ) %>%
    arrange(region, celltype, module_rank) %>%
    mutate(
      col_id = row_number(),
      display = factor(display, levels = display)
    )
}

column_order <- build_column_order()
column_order

In [ ]:
# ============================================================
# 8. Read GSEA files and select curated terms
# ============================================================

read_gsea_file <- function(path) {
  df <- readr::read_tsv(path, show_col_types = FALSE)

  required <- c("celltype_region", "module", "term", "nes", "fdr")
  missing <- setdiff(required, colnames(df))
  if (length(missing) > 0) {
    stop(path, " is missing columns: ", paste(missing, collapse = ", "))
  }

  parsed <- parse_celltype_region(df$celltype_region)

  df %>%
    mutate(
      module = as.character(module),
      key = paste(celltype_region, module, sep = "_"),
      celltype = parsed$celltype,
      region = parsed$region,
      term_clean = clean_go_term(term),
      term_match = normalize_term(term),
      nes = as.numeric(nes),
      fdr = as.numeric(fdr)
    ) %>%
    filter(nes > 0, fdr < 0.05)
}

collect_selected_hits <- function(go_file, kegg_file, column_order) {
  dfs <- list()

  if (file.exists(go_file)) {
    dfs[[length(dfs) + 1]] <- read_gsea_file(go_file)
  }
  if (file.exists(kegg_file)) {
    dfs[[length(dfs) + 1]] <- read_gsea_file(kegg_file)
  }
  if (length(dfs) == 0) {
    stop("No GO or KEGG input files found.")
  }

  df <- bind_rows(dfs) %>%
    filter(key %in% column_order$key)

  term_map <- purrr::imap_dfr(TERM_CATEGORIES, function(terms, category) {
    tibble(
      category = category,
      term_original = terms,
      term_match = normalize_term(terms),
      plot_term = pretty_term(terms)
    )
  }) %>%
    distinct(term_match, .keep_all = TRUE)

  df %>%
    left_join(term_map, by = "term_match") %>%
    filter(!is.na(category), !is.na(plot_term)) %>%
    mutate(category = factor(category, levels = names(TERM_CATEGORIES))) %>%
    group_by(category, plot_term, term_match, key) %>%
    summarise(
      best_fdr = min(fdr, na.rm = TRUE),
      max_nes = max(nes, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    mutate(
      score = -log10(pmax(best_fdr, 1e-300)),
      score = pmin(score, MAX_COLOR_VALUE)
    )
}

selected_hits <- collect_selected_hits(GO_FILE, KEGG_FILE, column_order)
selected_hits %>% count(category)

In [ ]:
# ============================================================
# 9. Build heatmap matrix and row order
# ============================================================

build_matrix_and_rows <- function(selected_hits, column_order) {
  row_info <- selected_hits %>%
    distinct(category, plot_term, term_match) %>%
    mutate(category = factor(category, levels = names(TERM_CATEGORIES))) %>%
    arrange(category, plot_term)

  mat <- selected_hits %>%
    select(term_match, key, score) %>%
    tidyr::complete(
      term_match = row_info$term_match,
      key = column_order$key,
      fill = list(score = 0)
    ) %>%
    group_by(term_match, key) %>%
    summarise(score = max(score), .groups = "drop") %>%
    pivot_wider(names_from = key, values_from = score, values_fill = 0) %>%
    arrange(match(term_match, row_info$term_match))

  mat_df <- as.data.frame(mat)
  rownames(mat_df) <- mat_df$term_match
  mat_df$term_match <- NULL
  mat_df <- mat_df[, column_order$key, drop = FALSE]

  list(mat = as.matrix(mat_df), row_info = row_info)
}

cluster_rows_within_category <- function(mat, row_info) {
  if (!CLUSTER_WITHIN_CATEGORY) {
    return(list(mat = mat, row_info = row_info))
  }

  new_order <- c()

  for (cat in names(TERM_CATEGORIES)) {
    idx <- which(as.character(row_info$category) == cat)

    if (length(idx) <= 2) {
      new_order <- c(new_order, idx)
      next
    }

    sub <- mat[idx, , drop = FALSE]

    if (all(apply(sub, 1, paste, collapse = ":") == paste(sub[1, ], collapse = ":"))) {
      new_order <- c(new_order, idx)
      next
    }

    hc <- tryCatch(
      hclust(dist(sub), method = "average"),
      error = function(e) NULL
    )

    if (is.null(hc)) {
      new_order <- c(new_order, idx)
    } else {
      new_order <- c(new_order, idx[hc$order])
    }
  }

  list(
    mat = mat[new_order, , drop = FALSE],
    row_info = row_info[new_order, , drop = FALSE]
  )
}

drop_empty_columns <- function(mat, column_order) {
  keep_cols <- colSums(mat, na.rm = TRUE) > 0

  if (!any(keep_cols)) {
    stop("All columns have zero enrichment after filtering curated terms.")
  }

  kept_keys <- colnames(mat)[keep_cols]

  column_order2 <- column_order %>%
    filter(key %in% kept_keys) %>%
    mutate(
      col_id = row_number(),
      display = factor(display, levels = display)
    )

  mat2 <- mat[, column_order2$key, drop = FALSE]

  list(mat = mat2, column_order = column_order2)
}

built <- build_matrix_and_rows(selected_hits, column_order)
clustered <- cluster_rows_within_category(built$mat, built$row_info)

# Remove candidate-module columns that have no positive curated GO/KEGG enrichment.
# This is done after building the curated-term matrix, so columns are kept only if
# at least one displayed term has score > 0.
nonempty <- drop_empty_columns(clustered$mat, column_order)

mat <- nonempty$mat
column_order <- nonempty$column_order
row_info <- clustered$row_info

message("Columns kept after removing empty columns: ", ncol(mat), " / ", ncol(clustered$mat))
row_info %>% count(category)


In [ ]:
# ============================================================
# 10. Convert matrix to ggplot data
# ============================================================

make_plot_data <- function(mat, row_info, column_order) {
  n_rows <- nrow(row_info)
  n_cols <- nrow(column_order)

  row_info2 <- row_info %>%
    mutate(
      row_id = row_number(),
      y = n_rows - row_id + 1,
      category = factor(category, levels = names(TERM_CATEGORIES))
    )

  heatmap_df <- as.data.frame(mat) %>%
    rownames_to_column("term_match") %>%
    pivot_longer(-term_match, names_to = "key", values_to = "score") %>%
    left_join(row_info2, by = "term_match") %>%
    left_join(column_order, by = "key") %>%
    mutate(x = col_id)

  category_ranges <- row_info2 %>%
    group_by(category) %>%
    summarise(
      ymin = min(y) - 0.5,
      ymax = max(y) + 0.5,
      ymid = (ymin + ymax) / 2,
      .groups = "drop"
    ) %>%
    mutate(
      xmin = CATEGORY_BAR_XMAX - CATEGORY_BAR_WIDTH,
      xmax = CATEGORY_BAR_XMAX
    )

  region_ranges <- column_order %>%
    group_by(region) %>%
    summarise(
      xmin = min(col_id) - 0.5,
      xmax = max(col_id) + 0.5,
      xmid = (xmin + xmax) / 2,
      .groups = "drop"
    ) %>%
    mutate(region = as.character(region))

  list(
    heatmap_df = heatmap_df,
    row_info = row_info2,
    column_order = column_order,
    category_ranges = category_ranges,
    region_ranges = region_ranges,
    n_rows = n_rows,
    n_cols = n_cols
  )
}

plot_data <- make_plot_data(mat, row_info, column_order)
str(plot_data, max.level = 1)

In [ ]:
# ============================================================
# 11. Plot functions
# ============================================================

make_heatmap_plot <- function(plot_data) {
  heatmap_df <- plot_data$heatmap_df
  row_info <- plot_data$row_info
  column_order <- plot_data$column_order
  category_ranges <- plot_data$category_ranges
  region_ranges <- plot_data$region_ranges
  n_rows <- plot_data$n_rows
  n_cols <- plot_data$n_cols

  celltype_bar_y <- n_rows + CELLTYPE_BAR_Y_OFFSET
  region_bar_y <- n_rows + REGION_BAR_Y_OFFSET
  region_text_y <- n_rows + REGION_TEXT_Y_OFFSET
  y_top <- n_rows + REGION_TEXT_Y_OFFSET + 0.5

  # First letter of each term starts at the heatmap right edge.
  # Heatmap tile centers are 1:n_cols, so right edge is n_cols + 0.5.
  term_x <- n_cols + 0.5 + TERM_LABEL_GAP
  x_right <- n_cols + RIGHT_TEXT_UNITS

  celltype_bar_df <- column_order %>%
    mutate(
      x = col_id,
      y = celltype_bar_y,
      celltype = factor(celltype, levels = CELLTYPE_ORDER)
    )

  region_bar_df <- region_ranges %>%
    mutate(
      ymin = region_bar_y - REGION_BAR_HEIGHT / 2,
      ymax = region_bar_y + REGION_BAR_HEIGHT / 2
    )

  x_label_df <- column_order %>%
    mutate(
      x = col_id,
      y = X_LABEL_Y,
      label = display
    )

  term_df <- row_info %>%
    transmute(
      x = term_x,
      y = y,
      plot_term = plot_term
    )

  category_legend_df <- tibble(
    category = names(TERM_CATEGORIES),
    label = category_label_short(names(TERM_CATEGORIES)),
    item = seq_along(names(TERM_CATEGORIES))
  ) %>%
    mutate(
      category = factor(category, levels = names(TERM_CATEGORIES)),
      x_box = term_x + CATEGORY_LEGEND_X_SHIFT,
      x_text = x_box + CATEGORY_LEGEND_TEXT_X,
      y = 0.5 - CATEGORY_LEGEND_TOP_GAP - (item - 1) * CATEGORY_LEGEND_ROW_GAP
    )

  y_bottom <- min(category_legend_df$y, x_label_df$y) - 0.8

  ggplot() +
    # Category color bar, attached directly to the heatmap left edge.
    # No category text on the left; category names are shown in the legend under terms.
    geom_rect(
      data = category_ranges,
      aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = category),
      color = NA
    ) +
    scale_fill_manual(values = category_colors, guide = "none") +

    ggnewscale::new_scale_fill() +

    # Main heatmap. geom_tile keeps PDF/SVG crisp; no raster imshow blur.
    geom_tile(
      data = heatmap_df,
      aes(x = x, y = y, fill = score),
      width = 1,
      height = 1,
      color = "#BDBDBD",
      linewidth = 0.45
    ) +
    scale_fill_gradientn(
      colors = c("#FFFFFF", "#FEE0D2", "#FC9272", "#DE2D26", "#A50F15"),
      limits = c(0, MAX_COLOR_VALUE),
      oob = scales::squish,
      name = expression(-log[10](FDR)),
      guide = guide_colorbar(
        order = 1,
        title.position = "right",
        barheight = unit(3.5, "in"),
        barwidth = unit(0.35, "in")
      )
    ) +

    ggnewscale::new_scale_fill() +

    # Cell type color strip. No text here; legend is on the left.
    geom_tile(
      data = celltype_bar_df,
      aes(x = x, y = y, fill = celltype),
      width = 1,
      height = CELLTYPE_BAR_HEIGHT,
      color = "white",
      linewidth = 0.35
    ) +
    scale_fill_manual(
      values = celltype_colors,
      name = "Cell type",
      guide = guide_legend(order = 2, override.aes = list(size = 8))
    ) +

    # Region annotation as a filled band above the cell type boxes.
    ggnewscale::new_scale_fill() +
    geom_rect(
      data = region_bar_df,
      aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax, fill = region),
      color = "white",
      linewidth = 0.35,
      show.legend = FALSE
    ) +
    scale_fill_manual(values = Region, guide = "none") +
    geom_text(
      data = region_ranges,
      aes(x = xmid, y = region_text_y, label = region, color = region),
      angle = REGION_TEXT_ANGLE,
      hjust = 0.5,
      vjust = 0.5,
      size = REGION_FONTSIZE / .pt,
      fontface = "bold",
      show.legend = FALSE
    ) +
    scale_color_manual(values = Region, guide = "none") +

    # Term labels: first letter starts right next to the heatmap right edge.
    geom_text(
      data = term_df,
      aes(x = x, y = y, label = plot_term),
      hjust = 0,
      vjust = 0.5,
      size = TERM_TEXT_SIZE / .pt,
      color = "black"
    ) +

    # Manually drawn x-axis labels, placed directly below the heatmap.
    geom_text(
      data = x_label_df,
      aes(x = x, y = y, label = label),
      angle = 90,
      hjust = 1,
      vjust = 0.5,
      size = XTICK_FONTSIZE / .pt,
      color = "black"
    ) +

    # Category legend under the last term, aligned with the term labels.
    ggnewscale::new_scale_fill() +
    geom_tile(
      data = category_legend_df,
      aes(x = x_box, y = y, fill = category),
      width = CATEGORY_LEGEND_BOX_SIZE,
      height = CATEGORY_LEGEND_BOX_SIZE,
      color = NA,
      show.legend = FALSE
    ) +
    geom_text(
      data = category_legend_df,
      aes(x = x_text, y = y, label = label),
      hjust = 0,
      vjust = 0.5,
      size = CATEGORY_LEGEND_TEXT_SIZE / .pt,
      fontface = "bold",
      lineheight = 0.85,
      color = "black",
      show.legend = FALSE
    ) +
    scale_fill_manual(values = category_colors, guide = "none") +

    scale_x_continuous(
      breaks = NULL,
      labels = NULL,
      limits = c(CATEGORY_BAR_XMAX - CATEGORY_BAR_WIDTH, x_right),
      expand = expansion(mult = c(0, 0))
    ) +
    scale_y_continuous(
      breaks = NULL,
      labels = NULL,
      limits = c(y_bottom, y_top),
      expand = expansion(mult = c(0, 0))
    ) +

    coord_fixed(ratio = 1, clip = "off") +

    labs(title = TITLE, x = NULL, y = NULL) +

    theme_minimal(base_size = 12) +
    theme(
      panel.grid = element_blank(),
      panel.background = element_blank(),
      plot.background = element_blank(),

      axis.text.x = element_blank(),
      axis.text.y = element_blank(),
      axis.ticks = element_blank(),

      plot.title = element_text(
        size = TITLE_FONTSIZE,
        face = "bold",
        hjust = 0.5,
        margin = margin(b = TITLE_BOTTOM_MARGIN)
      ),

      legend.position = "left",
      legend.box = "vertical",
      legend.title = element_text(size = LEGEND_FONTSIZE, face = "bold"),
      legend.text = element_text(size = LEGEND_FONTSIZE),
      legend.margin = margin(0, 10, 0, 0),

      plot.margin = margin(
        t = MAIN_MARGIN_TOP,
        r = MAIN_MARGIN_RIGHT,
        b = MAIN_MARGIN_BOTTOM,
        l = MAIN_MARGIN_LEFT
      )
    )
}

make_final_plot <- function(plot_data) {
  make_heatmap_plot(plot_data)
}



In [ ]:
# ============================================================
# 12. Preview plot in notebook
# ============================================================

final_plot <- make_final_plot(plot_data)
#final_plot

In [ ]:
# ============================================================
# 13. Save outputs
# ============================================================

save_outputs <- function(final_plot, plot_data) {
  n_rows <- plot_data$n_rows
  n_cols <- plot_data$n_cols

  fig_width <- max(
    MIN_FIG_WIDTH,
    LEFT_WIDTH + (n_cols + RIGHT_TEXT_UNITS) * CELL_SIZE
  )
  fig_height <- max(
    MIN_FIG_HEIGHT,
    TOP_HEIGHT + n_rows * CELL_SIZE + BOTTOM_HEIGHT+20
  )

  pdf_file <- file.path(OUTDIR, paste0(OUT_PREFIX, ".pdf"))
  png_file <- file.path(OUTDIR, paste0(OUT_PREFIX, ".png"))
  svg_file <- file.path(OUTDIR, paste0(OUT_PREFIX, ".svg"))

  # PDF: cairo_pdf usually gives clean vector output and editable text.
  tryCatch(
    {
      ggsave(
        pdf_file,
        plot = final_plot,
        width = fig_width,
        height = fig_height,
        units = "in",
        device = grDevices::cairo_pdf,
        limitsize = FALSE
      )
    },
    error = function(e) {
      message("cairo_pdf failed; falling back to standard pdf device.")
      ggsave(
        pdf_file,
        plot = final_plot,
        width = fig_width,
        height = fig_height,
        units = "in",
        device = "pdf",
        useDingbats = FALSE,
        limitsize = FALSE
      )
    }
  )

  ggsave(
    png_file,
    plot = final_plot,
    width = fig_width,
    height = fig_height,
    units = "in",
    dpi = 300,
    limitsize = FALSE
  )

  if (requireNamespace("svglite", quietly = TRUE)) {
    ggsave(
      svg_file,
      plot = final_plot,
      width = fig_width,
      height = fig_height,
      units = "in",
      device = svglite::svglite,
      limitsize = FALSE
    )
  } else {
    message("Package 'svglite' is not installed. Skipping SVG output.")
    svg_file <- NA_character_
  }

  if (WRITE_TABLES) {
    readr::write_tsv(column_order, file.path(OUTDIR, "candidate_module_column_order.tsv"))
    readr::write_tsv(selected_hits, file.path(OUTDIR, "selected_GO_KEGG_hits_for_heatmap.tsv"))
    readr::write_tsv(row_info, file.path(OUTDIR, "heatmap_row_order.tsv"))
    readr::write_tsv(as.data.frame(mat) %>% rownames_to_column("term_match"), file.path(OUTDIR, "heatmap_matrix_minus_log10_FDR.tsv"))
  }

  message("Saved files:")
  message(pdf_file)
  message(png_file)
  if (!is.na(svg_file)) message(svg_file)

  invisible(na.omit(c(pdf_file, png_file, svg_file)))
}

save_outputs(final_plot, plot_data)
